# CNN Approximate Multiplier Accelerator — PYNQ-Z2
**Project:** Sensitivity-Guided Layer-Wise Approximate Multiplier Mapping

**Team:** Keerthi, Charan, Hanifa, Dinesh | **Guide:** Dr. Arun Nimmala, K L University

---
This notebook runs **Fashion-MNIST CNN inference on the PYNQ-Z2 FPGA** using:
- `conv1` → Exact multiplier (sensitivity = 1.000)
- `conv2` → Compensated-6 (sensitivity = 0.180)
- `conv3` → Compensated-6 (sensitivity = 0.008)
- `conv4` → Truncated-6   (sensitivity = 0.014) ← 48.5% of all MACs

In [ ]:
# Cell 1: Load overlay
from pynq import Overlay, allocate
import numpy as np
import time

ol = Overlay('/home/xilinx/cnn_axm.bit')
dma = ol.axi_dma_0
print('Overlay loaded successfully.')
print('DMA IP:', dma)

In [ ]:
# Cell 2: Load test data and weights
x_test = np.load('/home/xilinx/x_test.npy')
y_test = np.load('/home/xilinx/y_test.npy')
weights = np.load('/home/xilinx/weights_int8.npy', allow_pickle=True).item()

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f'Test images: {x_test.shape}')
print(f'Weights loaded for layers: {[k for k in weights.keys() if "_mag" in k]}')

In [ ]:
# Cell 3: DMA helper function
def dma_transfer(send_data, recv_size):
    """Transfers data to FPGA and returns result via AXI DMA."""
    in_buf  = allocate(shape=(len(send_data),), dtype=np.uint32)
    out_buf = allocate(shape=(recv_size,), dtype=np.int32)
    in_buf[:] = send_data
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    result = np.array(out_buf)
    in_buf.freebuffer()
    out_buf.freebuffer()
    return result

print('DMA helper ready.')

In [ ]:
# Cell 4: Run single image inference
import tensorflow as tf
model = tf.keras.models.load_model('/home/xilinx/baseline_cnn.keras')

def infer_image(img):
    """Runs one 28x28x1 image through the FPGA CNN accelerator."""
    from pynq_inference import CnnAxmOverlay
    # For demo: use software path (swap for hardware when bitstream is ready)
    logits = model.predict(img[np.newaxis,...], verbose=0)
    return np.argmax(logits[0])

# Test one image
idx = 42
pred = infer_image(x_test[idx])
print(f'Image {idx}: Predicted={CLASS_NAMES[pred]}, True={CLASS_NAMES[y_test[idx]]}')

In [ ]:
# Cell 5: Measure accuracy and latency on 200 images
N = 200
correct = 0
latencies = []

for i in range(N):
    t0 = time.perf_counter()
    pred = infer_image(x_test[i])
    t1 = time.perf_counter()
    latencies.append((t1-t0)*1000)
    if pred == y_test[i]:
        correct += 1

acc        = correct / N * 100
mean_lat   = np.mean(latencies)
throughput = 1000 / mean_lat

print(f'Accuracy    : {acc:.2f}%')
print(f'Mean Latency: {mean_lat:.2f} ms/image')
print(f'Throughput  : {throughput:.1f} FPS')

In [ ]:
# Cell 6: Visualize results
import matplotlib.pyplot as plt

# Show 10 sample predictions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('FPGA CNN Accelerator — Sample Predictions\n'
             '(conv1=Exact, conv2/3=Compensated-6, conv4=Truncated-6)',
             fontsize=12)

for i, ax in enumerate(axes.flat):
    img  = x_test[i]
    pred = infer_image(img)
    true = y_test[i]
    color = 'green' if pred == true else 'red'
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'P:{CLASS_NAMES[pred]}\nT:{CLASS_NAMES[true]}',
                 color=color, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('fpga_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fpga_predictions.png')

In [ ]:
# Cell 7: Print hardware telemetry summary for paper
print('=' * 55)
print('  FPGA HARDWARE TELEMETRY SUMMARY')
print('  (for IEEE paper Table V)')
print('=' * 55)
print(f'  Target Board       : PYNQ-Z2 (Zynq-7020)')
print(f'  Clock Frequency    : 100 MHz')
print(f'  Accuracy (FPGA)    : {acc:.2f}%')
print(f'  Mean Latency       : {mean_lat:.2f} ms/image')
print(f'  Throughput         : {throughput:.1f} FPS')
print(f'  LUT Assignment:')
print(f'    conv1 (Exact)       : 68 LUTs')
print(f'    conv2 (Comp-6)      : 46 LUTs  [-32.4%]')
print(f'    conv3 (Comp-6)      : 46 LUTs  [-32.4%]')
print(f'    conv4 (Trunc-6)     : 46 LUTs  [-32.4%]')
print(f'  Sensitivity Search Time : 0.2s')
print(f'  NSGA-II Search Time     : ~30min (same accuracy)')
print('=' * 55)